# Task

Design an in-process logging library that an application can call to record events. Each call should fan out to one or more configured destinations.

### Clarifying Questions

Q: What kind of operations does the library need to support?

The library needs to support two main categories of operations:

Logging calls — the application submits a log record at a given severity level (DEBUG, INFO, WARN, ERROR, FATAL), along with a message and optionally structured metadata/context (e.g. key-value pairs, timestamps).

Configuration — at startup, the application sets up one or more destinations, each with its own minimum severity threshold and its own format. Once configured, the library routes each incoming log record to whichever destinations have a threshold at or below the record's severity level.

Q: So, the number of destinations will be same as the number of severity levels? Say 5 destinations for (DEBUG, INFO, WARN, ERROR, FATAL)?

No, the number of destinations is independent of the number of severity levels. You can configure any number of destinations (one, five, ten, etc.), and each destination simply has its own minimum severity threshold that determines which records it accepts.

Q: If a destination has minimum severity of FATAL, only fatal events will be logged?

Yes, that's correct. A destination with a minimum severity of FATAL will only receive records logged at the FATAL level.

Q: What happens if events of other severity are passed to it?

They are simply filtered out — that destination ignores any record whose severity level is below its configured minimum threshold.

Q: What formats do the destination support?

At minimum, the library should support two built-in formats: plain text (a human-readable string) and JSON (structured key-value output). The design should be open to additional formats being added later without modifying existing code.

Q: Do the destinations need to store the events in some data store?

The destinations write records to an output sink — for example, a file on disk, the console/stdout, or a database. The library's job is to get the formatted bytes to that sink; whether the sink itself persists data long-term is a concern of the sink, not the logging library.

Q: Is the sink pass to the library during intialization?

Yes, the sink (e.g. a file handle, stdout, a database connection) is provided to the library at initialization time as part of configuring each destination.

Q: How are the destinations being referenced? Do they need a unique ID or a name?

For v1, destinations don't need a unique ID or name — they are simply a collection configured at startup and referenced internally by the library. There's no requirement for the application to look up or modify a specific destination by name after initialization.

Q: Does the library need to handle concurrent calls?

Yes, concurrent calls must be handled safely. Specifically, when multiple threads log simultaneously to the same destination, the bytes of one record must never interleave with the bytes of another record on that destination.

Q: How about duplicate events? Is there any chance that the caller will send duplicate events that the library needs to figure out?

Duplicate detection is out of scope for this library. The logging service simply accepts every call the application makes and routes it to the appropriate destinations — it has no responsibility for identifying or deduplicating repeated records.

Q: How many callers does the library need to support simultaneously?

The library doesn't impose a fixed limit on the number of concurrent callers — it should handle whatever concurrency the host application produces. The key constraint is simply that writes to any given destination must be safe under concurrent access, as mentioned earlier.

Q: Is there a size limit on each event or limit on the length of the message and metadata?

No size limit on the message or metadata — the library should accept whatever the application passes in.

Q: What are things that are out of scope?

Here's what's explicitly out of scope for v1:
- Hot-reloading config at runtime — destinations and their settings are fixed after initialization.
- Async or buffered writes — all writes are synchronous in v1, though the design should accommodate async later.
- Remote/network destinations — no shipping logs over the network in v1, but the design should be open to it.
- Hierarchical or named loggers — there's no logger hierarchy (e.g. no parent/child logger relationships) and no named logger lookup.
- Duplicate detection — the library does not identify or suppress repeated records.
- Size limits — no enforcement of message or metadata size.

Q: How should the library respond when the sink fails during a write?

For v1, treat a sink write failure as a best-effort operation — the library should not crash the application, but there is no requirement to retry, buffer, or recover the failed record. How failures are surfaced (silently swallowed, logged to stderr, etc.) is a design decision left to you.

Q: if an event is going to multiple destination and the sink fails in one of the destination, should the other destinations continue with the operation?

Yes, a failure on one destination should not prevent the library from attempting to write to the remaining destinations — each destination is treated independently.

Q: Can you tell me how the Formatter is used?

Each destination has one formatter assigned to it at configuration time. When a log record arrives at a destination, the formatter is responsible for converting that record (severity level, message, metadata, timestamp, etc.) into the final byte representation — for example, a plain-text string or a JSON blob — before the destination writes those bytes to its sink.

# REQUIREMENTS

### My Answer - 1st attemp
1. The logging service logs events sent from applications to one or more destinations.
2. An event contains a severity level (DEBUG, INFO, WARN, ERROR, FATAL), and a message and optionally structured metadata/context (e.g. key-value pairs, timestamps).
3. The log event can be in either format of plain-text or JSON. The service will need to support additional formats in the future without any code modification.
4. Destinations are setup during initialization of the service. The service can support any number of destinations. 
5. Destination has configurations including a minimum severity threshold, format it supports and a sink where the logs will be output to.
6. An event will be written to one or more destinations as long as the event's severity is same or higher than the destination's minimum threshold.
7. The library needs to guarantee thread safety. When multiple threads log simultaneously to the same destination, the bytes of one record must never interleave with the bytes of another record on that destination.
8. There are no limits on the number of concurrent calls.


Out of Scope:
- Hot-reloading config at runtime — destinations and their settings are fixed after initialization.
- Async or buffered writes — all writes are synchronous in v1, though the design should accommodate async later.
- Remote/network destinations — no shipping logs over the network in v1, but the design should be open to it.
- Hierarchical or named loggers — there's no logger hierarchy (e.g. no parent/child logger relationships) and no named logger lookup.
- Duplicate detection — the library does not identify or suppress repeated records.
- Size limits — no enforcement of message or metadata size.

### Feedback
What went well

You captured the primary capabilities really well — logging calls with severity levels, fan-out to multiple destinations, per-destination thresholds, formats, and sinks are all clearly articulated. That's a solid foundation. 

Your out-of-scope section is excellent. You clearly internalized what you learned from the clarifying questions and translated it directly into boundaries, including async writes, network destinations, and hot-reload — all things that will keep your design focused. 
The thread safety requirement is spot-on. Calling out the non-interleaving guarantee specifically is exactly the right level of precision for a concurrency constraint. 

What needs improvement

One area that's missing is error handling at the destination level — specifically, what happens if a sink throws or fails during a write. Should the caller crash? Should other destinations still receive the record? This is worth capturing as a requirement because it directly affects how you'll structure the write loop in your design. 

You mentioned format is per-destination, but it's worth being explicit that format and destination type vary independently — any format can be paired with any sink. Right now requirement 5 reads as if a destination has 'a format it supports,' which could be interpreted as the sink dictating the format rather than the configuration choosing it freely. 

Minor suggestions

It'd be worth noting that each log record should capture a timestamp at the call site, not just rely on whatever the application passes in. This is a small but meaningful design decision that will come up when you define your record structure. 

### Sample Answer

Requirements:
1. Five severity levels: DEBUG < INFO < WARN < ERROR < FATAL
2. Each record carries timestamp, level, message, emitting thread name
3. Logger writes each record to one or more destinations, set at startup
4. Each destination has its own min-level threshold and its own format
   Format and destination type vary independently
5. Concurrent calls are safe. A record's bytes never interleave with
   another record's bytes on the same destination

Out of scope:
- Hot-reloading config at runtime
- Async / buffered writes
- Remote / network destinations in v1 (design should accommodate)
- Hierarchical / named loggers (com.app.service inheriting from com.app)


### My Answer - 2nd attempt

1. The logging service logs events sent from applications to one or more destinations.
2. An event contains a timestamp, name of the caller or application, a severity level (DEBUG, INFO, WARN, ERROR, FATAL), and a message and optionally structured metadata/context (e.g. key-value pairs, timestamps).
3. The log event can be in either format of plain-text or JSON. The service will need to support additional formats in the future without any code modification.
4. Destinations are setup during initialization of the service. The service can support any number of destinations. 
5. Destination has configurations including a minimum severity threshold, its own format and a sink where the logs will be output to. Format and destination type vary independently.
6. An event will be written to one or more destinations as long as the event's severity is same or higher than the destination's minimum threshold.
7. The library needs to guarantee thread safety. When multiple threads log simultaneously to the same destination, the bytes of one record must never interleave with the bytes of another record on that destination.
8. There are no limits on the number of concurrent calls.
9. If there is a sink write failure, the library and the calling application should not be affected. Also, the failure should not prevent the library from writing to other destinations.  


Out of Scope:
- Hot-reloading config at runtime — destinations and their settings are fixed after initialization.
- Async or buffered writes — all writes are synchronous in v1, though the design should accommodate async later.
- No retry, buffer, or recover the failed record during a sink write failure.
- Remote/network destinations — no shipping logs over the network in v1, but the design should be open to it.
- Hierarchical or named loggers — there's no logger hierarchy (e.g. no parent/child logger relationships) and no named logger lookup.
- Duplicate detection — the library does not identify or suppress repeated records.
- Size limits — no enforcement of message or metadata size.

### Feedback

What went well

You addressed the previous feedback on error handling really well — requirement 9 explicitly covers both that a sink failure shouldn't crash the caller and that it shouldn't block writes to other destinations. That's exactly the right level of precision. 
Your requirement 5 now explicitly states that format and destination type vary independently, which closes the ambiguity from last time about whether the sink dictates the format. That's a meaningful improvement. 
Your out-of-scope section is thorough and well-aligned with everything you learned from your clarifying questions — async writes, network destinations, hot-reload, and duplicate detection are all clearly bounded. 

Minor suggestions

One small thing worth noting: you mention the event includes a timestamp, but it would be worth clarifying that the timestamp is captured at the call site by the library itself, not just passed in by the caller. This is a subtle but important design decision that will come up when you define your log record structure. 

# Entities and Relationships

### My Answer - Entities 1st attempt
- Logger
- Event
- Destination

### Feedback
What went well

You correctly identified Logger, Event, and Destination as central pieces of the system — those are the right instincts and form a solid starting point. 
Recognizing Destination as its own entity shows you understand that each output target has its own configuration, which is a key part of this design. 
What needs improvement

Your list is missing a Formatter entity — this is the piece responsible for serializing a log event into plain-text or JSON, and it needs to be its own thing so you can swap formats independently of where logs go. 
You'd also want to pull out a Sink as its own entity — it's the interface that handles the actual byte-writing (console, file, etc.), and keeping it separate from Destination is what lets you add new output targets later without touching the rest of the system. 

Minor suggestions

Consider adding a LogLevel enum to represent the five severity levels — it gives you natural ordering for threshold comparisons and makes the severity concept explicit in code. 

### Sample Answer
- Logger
- LogRecord
- Destination
- Formatter (interface: PlainTextFormatter, JsonFormatter)
- Sink (interface: ConsoleSink, FileSink)
- LogLevel (enum: DEBUG, INFO, WARN, ERROR, FATAL)

### My Answer - Entities 2nd attempt
- Logger
- Event
- Destination
- Formatter (interface: PlainTextFormatter, JsonFormatter)
- Sink (interface: FileSink, ConsoleSink, DatabaseSink)
- LogLevel (enum: DEBUG, INFO, WARN, ERROR, FATAL)

### Feedback

What went well

You addressed the feedback from last time perfectly — you've added both Formatter (with PlainTextFormatter and JsonFormatter) and Sink (with FileSink, ConsoleSink, DatabaseSink) as explicit interfaces, which is exactly the right move for keeping format and output target independently swappable. 

Your LogLevel enum with all five severity levels (DEBUG, INFO, WARN, ERROR, FATAL) rounds out the list nicely and gives you a natural ordering for threshold comparisons. 
Logger, Event, and Destination are all present and correct, giving you a solid core to build the rest of the design around. 

### My Answer - Relationships
Logger class is the entry point of the library. During initialization, it takes in a list of destinations and keeps track of them. It exposes the log function which the applications will call to record events.  

Event class contains all the data related to the recorded event. 

Destination class contains the minimum severity threshold, a formatter, and a sink. 

### Feedback

What went well

You clearly identified Logger as the orchestrator — the entry point that holds the list of destinations and fans out log calls, which is exactly the right mental model. 
You nailed the composition inside Destination: it owns a Formatter and a Sink, which is the key insight that lets format and sink type vary independently without a class explosion. 
Your explanation that Logger creates an Event per log() call and passes it through to each Destination shows a clean understanding of the data flow across the whole system. 

### Sample Answer

"Logger is the orchestrator and the only class the application talks to. It owns an immutable list of Destinations, fixed at construction. On every log() call, Logger captures the timestamp and current thread's name, builds a LogRecord, and iterates over destinations handing each one the same record. Destination is where the per-target work lives - it composes a Formatter (for serialization) and a Sink (for actual byte-writing), holds its own minimum level threshold, and owns the per-destination lock. LogRecord is an immutable value object that flows through the system. Formatter and Sink are interfaces with concrete implementations (PlainText/JSON, Console/File), and any combination is just a constructor argument. There are no back-references and no shared mutable state across destinations - each one protects its own resource."


# Class Design
```
enum LogLevel:
    DEBUG, INFO, WARN, ERROR, FATAL

class Logger:
    - _destinations: list<Destination>
    + Logger(destinations: list<Destination>)
    + Log(log_level: LogLevel, message: string, metadata: Map<string, string>)

class Event:
    - _timestamp: Datetime
    - _caller_name: string
    - _log_level: LogLevel
    - _message: string
    - _metadata: Map<string, string> 
    + Event(timestamp: Datetime, caller_name: string, log_level: LogLevel, message: string, metadata: Map<string, string>)
    + getTimestamp() -> Datetime
    + getCallerName() -> string
    + getLogLevel() -> LogLevel
    + getMessage() -> string
    + getMetadata() -> Map<string, string>

class Destination:
    - _min_level_threshold: LogLevel
    - _formatter: Formatter
    - _sink: Sink
    - _lock: Lock
    + Destination(min_level_threshold: LogLevel, formatter: Formatter, sink: Sink)
    + WriteEvent(event: Event)

abstract class Formatter:
    + Formatter()
    + Convert(event: Event) -> list<Bytes>

class PlainTextFormatter implements Formatter:
    + PlainTextFormatter()
    + Convert(event: Event) -> list<Bytes>

class JsonFormatter implements Formatter:
    + JsonFormatter()
    + Convert(event: Event) -> list<Bytes>

abstract class Sink:
    + Sink()
    + Write(bytes)

class ConsoleSink implements Sink:
    + ConsoleSink()
    + Write(bytes)

```

### My Answer

A List of destinations are passed to the Logger class during initialization and the logger class keeps track of the list throughout its lifecycle. Applications will call the Log() function with information about the event. The Log() will capture the current time and name of the caller as event comes in. Then creates an Event object and passes it to all the destinations.

It is up to the destinations to determine whether to accept the event based on its own minimum LogLevel threshold. If the event is same or above the threshold, the destination will call its own Formatter to convert the event to the right format bytes and output to the sink. Each destination class has its own lock which prevents concurrent writes to the sink.  

I have included the abstract class Formatter and the implementation classes PlainTextFormatter and JsonFormatter. Same for the abstract class Sink, I included the implementation class ConsoleSink. There could be other implementations such as DatabaseSink or FileSink which I didn't include here. In the future, new formats or sinks could be supported by adding new implementations. No code modifications are necessary in the library or existing classes.

### Feedback
What went well

Your composition of Formatter and Sink as independent interfaces on Destination is exactly right — it means any format can pair with any sink without a class explosion, which was the key design challenge here. 
Your Destination class is one concrete class that composes _formatter, _sink, and _min_level_threshold, and your verbal explanation of the filter-then-format-then-write flow inside WriteEvent() shows you understand the pipeline clearly. 
Your Event is a clean immutable value object — all state, getters only, no setters or business logic leaking in. 

Minor suggestions

You might consider adding convenience methods like debug(), info(), warn(), error(), and fatal() on Logger — they're one-liners that delegate to Log(), but they make the calling code much more readable and are a common expectation for logging APIs. 

### Sample Answer
"This design separates concerns cleanly. Logger orchestrates dispatch, LogRecord is the immutable unit of data, LogLevel is the five-valued enum, Formatter is the serialization interface, Sink is the output interface, and Destination composes them. Pulling Sink out as its own interface (rather than using inheritance on Destination) is what gives the requirement-stated future remote destination a clean place to land - a RemoteSink slots in next to ConsoleSink and FileSink with no other class touched."

```
enum LogLevel:
    DEBUG
    INFO
    WARN
    ERROR
    FATAL
    // ordered: DEBUG < INFO < WARN < ERROR < FATAL

class LogRecord:
    - timestamp: Instant
    - level: LogLevel
    - message: String
    - threadName: String

    + LogRecord(timestamp, level, message, threadName)
    + getters for all fields

interface Formatter:
    + format(record: LogRecord) -> String

class PlainTextFormatter implements Formatter
class JsonFormatter implements Formatter

interface Sink:
    + write(formatted: String)

class ConsoleSink implements Sink
class FileSink implements Sink:
    - filePath: String

class Destination:
    - formatter: Formatter
    - minLevel: LogLevel
    - sink: Sink

    + Destination(formatter, minLevel, sink)
    + write(record: LogRecord)

class Logger:
    - destinations: List<Destination>

    + Logger(destinations: List<Destination>)
    + log(level: LogLevel, message: String)
    + debug(message: String)
    + info(message: String)
    + warn(message: String)
    + error(message: String)
    + fatal(message: String)

```

Implement Log(log_level, message, metadata) on Logger. Focus on: where and how you capture the timestamp and caller name, how you construct the Event, and how you fan it out to all destinations. Also walk me through what happens if one destination throws — does it affect the others?

### My Answer
Upon the method being called, I will get the current timestamp in UTC and the caller thread's name. Then create an immutable event object with these information. 

I then pass the event to each destination which is wrapped in a try-catch block. If any of them throws an error, the Logger and other destinations will not be affected.

This publisher/subscriber pattern is simple to implement, highly extensible and good for a light-weight logger. However, the calls to the destinations are synchronous and sequential. 

If there are a lot of destinations or if any of the destination is slow to respond, the overall performance will be impacted. In that case, I can change to dispatcher approach by using asynchronous queue.    

### Feedback
What went well

You capture the timestamp and thread name once at the top of Log(), so every destination sees the same moment for a single call — exactly right. 
Wrapping each dest.WriteEvent(event) in its own try/except correctly isolates failures so one bad destination never affects the others or the calling application. 
You correctly kept level filtering out of the Logger and let it live per-destination, which supports the per-destination thresholds the requirements demand. 

Minor suggestions

Consider emitting your diagnostic to stderr rather than print() (stdout), since a ConsoleSink might also be writing to stdout and you don't want your own error noise mixed into the log stream. 

```
import sys

# Standard Error (Sent to stderr)
print("ERROR: Something went wrong!", file=sys.stderr)

# Alternative: Writing directly to the stream without print's automatic newline
sys.stderr.write("Direct error write\n")
```



In [ ]:
class Logger:
    def __init__(self, destinations: list[Destination]):
        self._destinations = destinations
    
    def Log(self, log_level: LogLevel, message: str, metadata: dict[str, str]):
        event = Event(datetime.now(UTC), threading.current_thread().name, log_level, message, metadata)
        for dest in self._destinations:
            try:
                dest.WriteEvent(event)
            except Exception as e:
                print(f"Error occurs when outputing to destination: {e}")

Implement WriteEvent(event) on Destination. Walk me through the full pipeline: how you decide whether to accept the event, how you get formatted bytes, and how you ensure two threads writing simultaneously never interleave bytes on the same sink. Where exactly does your lock live, and what does your error handling look like if _sink.Write() throws?

### My Answer
The WriteEvent() will first check if the value for the event's LogLevel is less than the destination's minimum threshold. If it is, the event will be rejected (return, nothing happens). Otherwise, it will format the event and write it to the sink. Both of these operations are inside per-destination lock, so that no two threads can write simultaneously to the sink.  

No error handling is needed around the _sink.Write() because the Logger already has a try-catch block on destination.WriteEvent(). However, if the destination has its own special handling for sink errors or wants to output to a backup alternative, a try-catch block could be added on the _sink.Write().  

### Feedback
What went well

Your level threshold check happens first with a silent return below threshold, exactly the right gate before doing any work. 
Your per-destination lock lives on the Destination instance, so each sink has its own lock and bytes never interleave under contention. 

Minor suggestions

You're formatting inside the lock, but since events are immutable and formatters are pure, you can move self._formatter.Convert(event) outside the lock to shorten the critical section and reduce contention. 

Consider whether Destination should catch its own sink errors so one bad sink never propagates — relying solely on Logger's try/catch works, but local handling keeps each destination self-contained for future backup/fallback logic. 

In [ ]:
class Destination:
    def __init__(self, min_level_threshold: LogLevel, formatter: Formatter, sink: Sink):
        self._min_log_level_threshold = min_level_threshold
        self._formatter = formatter
        self._sink = sink
        self._lock = threading.Lock()

    def WriteEvent(self, event: Event):
        if event.getLogLevel().value < self._min_log_level_threshold.value:
            return

        with self._lock:
            formatted_event = self._formatter.Convert(event)
            self._sink.Write(formatted_event)
            

# Extensibility


#### Q: Right now, your Logger.Log() method calls dest.WriteEvent(event) synchronously inside the for-loop, so the calling thread blocks until every destination has finished writing. If one of those destinations is slow — say a future FileSink writing to a network-mounted drive — every thread in the application stalls waiting for it. How would you redesign Destination (or the relationship between Logger and Destination) to make writes non-blocking, while still guaranteeing that a single destination never interleaves bytes from two concurrent log calls?

### My Answer

I will change the Destination class to have its own internal queue in which events are inserted and pending for process. Each destination will have its own worker thread processing the events in the queue.  
When the Logger calls the destination's WriteEvent(), the event is added to the internal queue and the function returns immediately. Destination's worker thread will pick up the events from the queue and process them individually. This way, the writes become non-blocking. Since python's build-in queue is thread-safe, two concurrent log calls cannot write to the queue at the same time. Since there is only one worker thread per destination processing events one at a time, there will not be concurrent writes to the sink. 

There is no change in the Logger class. All changes are within the Destination class.

### Feedback

What went well

You correctly identified that all changes belong inside the Destination class — adding an internal queue and a worker thread there — while Logger stays exactly the same. That's precisely the right encapsulation: the caller shouldn't care whether a destination writes synchronously or asynchronously. 
Your reasoning about thread safety is solid: you noted that Python's built-in queue handles concurrent enqueues safely, and that having a single worker thread per destination eliminates the need for a lock on the sink. That's a clean insight — you replaced one synchronization mechanism with a simpler architectural guarantee. 

Minor suggestions

One thing worth thinking about for a follow-up: what happens to events already sitting in the queue when the application shuts down? You'd want to decide whether the worker thread drains the queue before exiting (so no logs are lost) or stops immediately. This is called a 'shutdown/drain policy' and is something interviewers love to ask about for async designs like this. 

####  Q: Suppose your team wants to add log filtering beyond the simple minimum-level threshold that Destination currently enforces — for example, 'only write events whose metadata contains the key "request_id"', or 'suppress any DEBUG events whose message matches a noisy regex'. Where in your current design would you add this capability, and what interface or class would you introduce so that new filter rules can be added later without modifying Destination or Formatter?

### My Answer

I will add an interface or abstract class EventFilter which has abstract method RejectEvent(event). This allows each implementation to have different log filtering rules. A list of implementations will be passed to the Destination class during initialization. In the Destination.WriteEvent(), the event will be passed to the RejectEvent() of each of these filter implementations. If any of them returns true, the event will not be logged. 
In the future more implementations can be added without modifying existing code.  

### Feedback

What went well

You correctly identified that Destination is the right place to apply filters — it already owns the 'should I write this event?' responsibility, so adding filter evaluation there keeps the design coherent without spreading concerns around. Your WriteEvent() update shows exactly where the check fits in the flow: after the level threshold check, before formatting and writing. 

Your EventFilter abstract class is a clean, minimal interface that makes the open/closed principle concrete — new filtering rules (metadata key checks, regex suppression, etc.) can be added as new subclasses without touching Destination, Formatter, or Sink at all. Passing the list at construction time also keeps it consistent with how the rest of your Destination is configured. 

Minor suggestions

One small thing to think about: since filters are per-Destination, you have a lot of flexibility — but if you find yourself needing a filter that applies to all destinations (e.g., a global 'suppress this noisy caller entirely'), you might consider whether Logger.Log() could also hold a list of global pre-filters evaluated before fan-out. Not necessary now, just a useful evolution path to keep in mind. 

In [ ]:
class EventFilter(ABC):
    def __init__(self):
        pass

    @abstractmethod
    def RejectEvent(event: Event) -> bool:
        pass

class Destination:
    def __init__(self, min_level_threshold: LogLevel, formatter: Formatter, sink: Sink, filters: list[EventFilter]):
        self._min_log_level_threshold = min_level_threshold
        self._formatter = formatter
        self._sink = sink
        self._filters = filters
        self._lock = threading.Lock()

    def WriteEvent(self, event: Event):
        if event.getLogLevel().value < self._min_log_level_threshold.value:
            return

        for filter in self._filters:
            if filter.RejectEvent(event):
                return

        formatted_event = self._formatter.Convert(event)
                        
        with self._lock:
            try:
                self._sink.Write(formatted_event)
            except Exception as e:
                sys.stderr.write("Error occurs when writing to sink:\n")
                sys.stderr.write(f"{e}\n")


#### Q: Your Formatter.Convert() returns list<Bytes>, and Sink.Write() accepts those bytes — which is a clean separation. Now imagine adding a FileSink that needs to rotate the log file when it reaches 100 MB. Rotation requires: closing the current file handle, renaming it, and opening a new one — all while concurrent log calls may be in flight. Given that the _lock currently lives in Destination (not in Sink), walk me through exactly where the rotation logic lives, who holds the lock during rotation, and whether any changes to the Sink interface are needed.

### My Answer
The rotation logic should be inside the FileSink, since only the sink itself has the handle to the file and knows how big the file has reached. 
If I reuse the earlier design having an asynchronous queue inside the destination and a single worker thread per destination, there is no need for a lock. When the sink is busy rotating files, any concurrent incoming events will go to the queue and not being affected.    

There are no changes needed on the Sink interface. 

### Feedback
What went well

You correctly identified that rotation logic belongs inside FileSink — your reasoning is spot-on: FileSink is the only class that owns the file handle and knows the current file size, so it's the natural home for rotation. This shows good understanding of the Single Responsibility Principle. 

You correctly noted that the Sink interface doesn't need to change, which is exactly right — rotation is an internal implementation detail of FileSink, not something the interface needs to expose. Your existing Write(bytes) contract stays intact. 

Minor suggestions

One thing worth thinking through: in the current design where the lock lives in Destination, the lock is acquired before sink.Write() is called — so while one thread is inside FileSink.Write() doing the rotation (closing, renaming, reopening the file handle), other threads are blocked at the Destination level waiting for the lock. This actually works correctly as-is! Your async queue idea is a valid future enhancement, but it's worth recognizing that the current synchronous design already handles this safely without any changes.

# Final Implementation

In [ ]:
from datetime import datetime, UTC
from enum import Enum
from abc import ABC, abstractmethod 
import threading
import sys

class LogLevel(Enum):
    DEBUG = 0, 
    INFO = 1, 
    WARN = 2, 
    ERROR = 3, 
    FATAL = 4

class Event:
    def __init__(self, timestamp: datetime, caller_name: str, log_level: LogLevel, message: str, metadata: dict[str, str]):
            self._log_level = log_level

    def getLogLevel(self) -> LogLevel:
        return self._log_level

class Formatter(ABC):
    def __init__(self):
        pass

    @abstractmethod
    def Convert(event: Event) -> list[bytes]:
        pass

class Sink(ABC):
    def __init__(self):
        pass

    @abstractmethod
    def Write(output: bytes):
        pass

class EventFilter(ABC):
    def __init__(self):
        pass

    @abstractmethod
    def RejectEvent(event: Event) -> bool:
        pass

class Destination:
    def __init__(self, min_level_threshold: LogLevel, formatter: Formatter, sink: Sink, filters: list[EventFilter]):
        self._min_log_level_threshold = min_level_threshold
        self._formatter = formatter
        self._sink = sink
        self._filters = filters
        self._lock = threading.Lock()

    def WriteEvent(self, event: Event):
        if event.getLogLevel().value < self._min_log_level_threshold.value:
            return

        for filter in self._filters:
            if filter.RejectEvent(event):
                return

        formatted_event = self._formatter.Convert(event)
                        
        with self._lock:
            try:
                self._sink.Write(formatted_event)
            except Exception as e:
                sys.stderr.write("Error occurs when writing to sink:\n")
                sys.stderr.write(f"{e}\n")


class Logger:
    def __init__(self, destinations: list[Destination]):
        self._destinations = destinations
    
    def Log(self, log_level: LogLevel, message: str, metadata: dict[str, str]):
        event = Event(datetime.now(UTC), threading.current_thread().name, log_level, message, metadata)
        for dest in self._destinations:
            try:
                dest.WriteEvent(event)
            except Exception as e:
                sys.stderr.write("Error occurs when outputing to destination:\n")
                sys.stderr.write(f"{e}\n")


# Key Takeaways

### Requirements
You demonstrated solid progression across two attempts, quickly addressing feedback on error handling and format independence, though a subtle but important detail about timestamp ownership kept appearing across both responses.

When a system fans out to multiple destinations, sink failures should be isolated. If one destination fails during a write, the logger should catch that error silently (or log it internally) and continue writing to the remaining destinations. The caller should never crash because one sink had a problem. This is a key resilience requirement to call out explicitly in your requirements gathering phase.

In a logging system, the timestamp should be captured by the library itself at the call site, not passed in by the caller. If the caller provides the timestamp, it can be wrong, delayed, or missing entirely. The library owning the timestamp ensures consistency and accuracy across all log records. When you define your LogRecord or LogEvent data structure, show that the library sets this field automatically.

### Entities
You started with a solid core but were missing key entities on your first attempt, then quickly corrected course to produce a clean, well-structured logging system design by your second try.

Separate Formatter and Sink into their own interfaces in a logging system. Formatter handles serializing a log event into a specific format (plain text, JSON), while Sink handles the actual writing to an output target (file, console, database). Keeping them separate means you can mix and match any format with any output target without creating a new class for every combination.

Use composition inside a Destination class to hold a Formatter and a Sink. This is the 'has-a' relationship in action — a Destination has a Formatter and has a Sink. This avoids a class explosion where you would otherwise need classes like JsonFileSink, PlainTextConsoleSink, and so on for every combination.

Model severity levels as an enum with a natural ordering (DEBUG, INFO, WARN, ERROR, FATAL). Using an enum makes severity an explicit, comparable value in code, which lets you do threshold checks like 'only log events at WARN or above' with a simple comparison instead of string matching.

### Class Design
You demonstrated a strong grasp of composition, interface design, and clean value objects, producing a solid logging system design in a single attempt with only a minor API convenience gap.

Logging APIs commonly expose convenience methods like debug(), info(), warn(), error(), and fatal() on the Logger class. These are simple one-liners that call the main Log() method with a preset level, but they make calling code much cleaner and are a standard expectation interviewers look for in a polished logging design.

### Implementation
You demonstrated solid fundamentals across both attempts, with the feedback being mostly minor refinements rather than major gaps, but there are a couple of concrete optimization patterns worth memorizing before your real interview.

Move pure, side-effect-free work outside of locks to shorten the critical section. If formatting an event does not depend on shared mutable state, do it before acquiring the lock. Shorter critical sections mean less contention and better throughput under load.

Each destination or sink should catch and handle its own errors internally rather than relying on the caller to wrap it in a try/except. This keeps each component self-contained and makes it easier to add fallback or retry logic later without changing the Logger.

When writing diagnostic or error messages about your logging system itself, write to stderr instead of stdout. A ConsoleSink already writes to stdout, so mixing your own error noise into that same stream makes it hard to separate real log output from internal errors.

### Extensibility
You demonstrated strong OOP instincts across all three attempts, correctly placing logic in the right classes and keeping interfaces stable, with the main gaps being around async lifecycle management and global vs per-destination concerns.

When designing async queues with worker threads, always define a shutdown/drain policy. This means deciding whether the worker thread finishes processing all queued items before stopping, or stops immediately. Interviewers expect you to address this because data loss on shutdown is a real production concern.

The Open/Closed Principle means a class should be open for extension but closed for modification. A clean way to apply this is with an abstract filter or strategy class, where new behaviors (like new log filters) are added as new subclasses rather than by editing existing classes like Destination or Sink.

When a lock is acquired at a higher level (like in Destination) before calling a lower-level method (like FileSink.Write), the lower-level method is already protected from concurrent access. You do not need a second lock inside FileSink because only one thread can be inside Write at a time. Recognizing where thread safety is already guaranteed saves you from over-engineering.

Per-destination filters and global pre-filters solve different problems. Per-destination filters control what each sink receives. A global pre-filter on the Logger itself stops an event before it fans out to any destination at all. Knowing this distinction lets you choose the right level of filtering for a given requirement.

### Overall Performance

```
Overall Score
90%

Peer Average
80%

Your Average
92%
```
